# 🫁 ChestNet on Colab: Part 3, EfficientNet-B0 / B3 (A → Z)

This notebook runs **the whole EfficientNet part** of ChestNet, from raw X-rays to report-ready figures. It uses the shared `src/` code, so your results follow exactly the same rules as the other three models.

```text
🅰️ Setup → 📦 Data → ✂️ Same splits as the team → 🔍 Pipeline check → 🧠 Meet EfficientNet
   → 🔬 Sanity checks → 🚀 Smoke test → 🏋️ Train B0 → 🏋️ Train B3 → 🧪 (optional) B3 @ 300 px
   → 📈 Curves → 🎚️ Threshold tuning (validation) → 🧾 Test once → 🔥 Grad-CAM → 🏁 Scoreboard → 💾 Save   🆉
```

### 🗺️ The three EfficientNet runs

| Run | Config | Image size | Role |
|---|---|---|---|
| 🟩 **B0** | `configs/efficientnet.yaml` | 224 | **Official entry** in the 4-model comparison |
| 🟩 **B3** | `configs/efficientnet_b3.yaml` | 224 | Same rules, bigger network: "does a bigger EfficientNet help?" |
| 🧪 **B3 @ 300** | `configs/efficientnet_b3_300.yaml` | 300 | **Ablation only**: "does the *resolution* part of compound scaling help?" |

**⏱️ Time on a T4:** B0 ≈ 25–40 min, B3 ≈ 45–70 min, B3 @ 300 ≈ 1.5 h (optional). Results are copied to Drive after every run, so a disconnect never loses a finished model.

**Before you start:** run notebook 01 once (it creates the shared splits), push the EfficientNet code to your branch, add the Kaggle secrets, and pick **Runtime → T4 GPU**.

## 1️⃣ GPU, disk and settings 🖥️

In [ ]:
import os, shutil, subprocess, json, time
from pathlib import Path

gpu = subprocess.run(["nvidia-smi", "--query-gpu=name,memory.total", "--format=csv,noheader"],
                     capture_output=True, text=True).stdout.strip()
print("🎮 GPU:", gpu or "⚠️ NONE – Runtime > Change runtime type > T4 GPU")

REPO_URL  = "https://github.com/Deep-Learning-Models/ChestNet.git"
BRANCH    = "main"                                   # 👈 your EfficientNet branch until it is merged
REPO_DIR  = "/content/ChestNet"
DRIVE_DIR = "/content/drive/MyDrive/ChestNet_colab"  # permanent storage (shared with notebook 01)

from google.colab import drive, userdata
drive.mount("/content/drive")
os.makedirs(DRIVE_DIR, exist_ok=True)
CACHE_TAR = f"{DRIVE_DIR}/chestnet_cache.tar"

def secret(name):
    try:
        return userdata.get(name)
    except Exception:
        return None

## 2️⃣ Clone the code and install packages 📦

In [ ]:
token = secret("GITHUB_TOKEN")
url = REPO_URL.replace("https://", f"https://{token}@") if token else REPO_URL
if not os.path.exists(REPO_DIR):
    !git clone -q --branch {BRANCH} {url} {REPO_DIR}
else:
    !git -C {REPO_DIR} pull -q
%cd {REPO_DIR}
!git log --oneline -3
!pip install -q "keras-hub>=0.18" "kaggle>=1.6"

# Make sure this copy has the EfficientNet B0/B3 code
for f in ["configs/efficientnet_b3.yaml", "src/tune_thresholds.py", "src/efficientnet_report.py"]:
    assert Path(f).exists(), f"❌ {f} missing – push the EfficientNet code to {BRANCH} first"
import tensorflow as tf, keras
print("✅ TensorFlow", tf.__version__, "| Keras", keras.__version__, "| GPUs:", tf.config.list_physical_devices("GPU"))

## 3️⃣ Get the X-rays ⬇️
Same download as notebook 01 (Kaggle sample, 5,606 images → `data/raw/images/`, labels → `data/sample_labels.csv`).

In [ ]:
import zipfile, pandas as pd
api_token = secret("KAGGLE_API_TOKEN")
if api_token:
    os.environ["KAGGLE_API_TOKEN"] = api_token
else:
    os.environ["KAGGLE_USERNAME"] = secret("KAGGLE_USERNAME") or ""
    os.environ["KAGGLE_KEY"] = secret("KAGGLE_KEY") or ""

DL_DIR, img_dir = Path("/content/nih_sample"), Path("data/raw/images")
if len(list(img_dir.glob("*.png"))) < 5606:
    if not list(DL_DIR.rglob("*.png")):
        !kaggle datasets download nih-chest-xrays/sample -p {DL_DIR} --unzip
    for z in DL_DIR.rglob("*.zip"):
        zipfile.ZipFile(z).extractall(z.parent); z.unlink()
    shutil.copy(next(DL_DIR.rglob("sample_labels.csv")), "data/sample_labels.csv")
    img_dir.mkdir(parents=True, exist_ok=True)
    for p in DL_DIR.rglob("*.png"):
        if not (img_dir / p.name).exists():
            shutil.move(str(p), img_dir / p.name)
    shutil.rmtree(DL_DIR)
print(f"✅ {len(list(img_dir.glob('*.png')))} X-rays ready")

# ♻️ Restore the resized-image cache made in notebook 01 (much faster first epoch)
if os.path.exists(CACHE_TAR):
    !tar -xf "{CACHE_TAR}" -C {REPO_DIR}
    print("✅ 224 px cache restored")

## 4️⃣ Use the SAME patient-level splits as the team ✂️
🧍 **Why it matters:** if EfficientNet were tested on different patients than ResNet50 or ViT, the comparison would be unfair. The splits are committed in `data/splits/`; we only create them if they are missing (same seed → same result).

⚠️ **Beginner trap — data leakage:** a patient often has several X-rays. If one is in *train* and another in *test*, the model can "recognise the patient" instead of the disease, and test scores look better than reality. `split_report.json` must say **0 shared patients**.

In [ ]:
from src.config import load_config
from src.preprocessing.splits import load_splits
if not Path("data/splits/train.csv").exists():
    print("🆕 No committed splits found – creating them (seed 42)")
    !python -m src.preprocessing.splits
rep = json.load(open("data/splits/split_report.json"))
assert rep["patient_overlap_total"] == 0 and rep["image_overlap_total"] == 0, "❌ leakage between splits!"
display(pd.read_csv("data/splits/summary.csv", index_col=0))
print(f"✅ No leakage: {rep['train_patients']} / {rep['val_patients']} / {rep['test_patients']} patients (train / val / test)")

## 5️⃣ Verify the shared rules ✅
Checks every model config (including EfficientNet) against the locked protocol, the input pipeline and augmentation.

In [ ]:
!python -m src.check_pipeline

## 6️⃣ Meet EfficientNet 🧠

🏗️ **Analogy:** imagine upgrading a camera. You could add more lenses (*depth*), make the sensor wider (*width*) or take sharper photos (*resolution*). Doing just one hits a wall fast. EfficientNet's **compound scaling** upgrades all three in a fixed ratio with one knob φ:

| | depth | width | resolution | params (backbone) |
|---|---|---|---|---|
| **B0** (φ = 0) | 1.0× | 1.0× | 224 px | ~4.0 M |
| **B3** (φ ≈ 3) | ~1.8× | ~1.4× | 300 px | ~10.8 M |

🧱 Its building block, **MBConv**, is: 1×1 expand → *depthwise* 3×3/5×5 conv (one filter per channel = very cheap) → **Squeeze-and-Excitation** (the block learns *which channels matter*, like turning up the volume on useful features) → 1×1 project + skip connection.

📥 **Input rule:** our pipeline gives raw **0–255** pixels and Keras EfficientNet normalises *inside* the model. Dividing by 255 yourself is a classic mistake: the pretrained filters would see almost-black images.

Let's build both and look at them 👇

In [ ]:
from src.models import build_model
from src.preprocessing.dataset import make_dataset

models = {}
for run, path in [("B0", "configs/efficientnet.yaml"), ("B3", "configs/efficientnet_b3.yaml")]:
    cfg = load_config(path)
    m = build_model(cfg)
    bn = [l for l in m.layers if isinstance(l, keras.layers.BatchNormalization)]
    trainable = int(sum(keras.ops.size(w) for w in m.trainable_weights))
    print(f"🟩 {run}: {m.count_params()/1e6:.2f} M params ({trainable/1e6:.2f} M trainable) | "
          f"{len(m.layers)} layers | BN frozen: {sum(not l.trainable for l in bn)}/{len(bn)} | "
          f"Grad-CAM layer output: {m.get_layer('top_activation').output.shape}")
    models[run] = (cfg, m)

## 7️⃣ Sanity checks before spending GPU time 🔬
Three quick tests that catch most beginner bugs:
1. **Pixel range** of a training batch is 0–255 (not 0–1).
2. **Outputs** are 8 independent probabilities in [0, 1] (sigmoid, multi-label) — they do **not** sum to 1.
3. **Starting loss** is sensible and the model can **overfit one batch** (if it can't learn 32 images, it can't learn 3,000 — a bug somewhere).

In [ ]:
from src.preprocessing.dataset import compute_pos_weights
from src.utils.losses import weighted_bce

cfg, model = models["B0"]
train_df, val_df, test_df = load_splits(cfg)
x, y = next(iter(make_dataset(train_df, cfg, training=True, split="train")))
print(f"1️⃣ batch {tuple(x.shape)} pixels {float(tf.reduce_min(x)):.0f}–{float(tf.reduce_max(x)):.0f}  (expect 0–255)")
p = model(x, training=False).numpy()
print(f"2️⃣ outputs {p.shape}, range {p.min():.2f}–{p.max():.2f}, row sums ≈ {p.sum(1)[:3].round(2)} (NOT 1 → multi-label ✅)")

# 3️⃣ overfit ONE batch for 30 steps on a throw-away copy
probe = build_model(cfg)
probe.compile(keras.optimizers.Adam(1e-3), loss=weighted_bce(compute_pos_weights(train_df, cfg["data"]["classes"])))
losses = [float(probe.train_on_batch(x, y)) for _ in range(30)]
print(f"3️⃣ loss {losses[0]:.3f} → {losses[-1]:.3f}", "✅ it learns" if losses[-1] < 0.5 * losses[0] else "⚠️ not dropping – check the setup")
del probe

## 8️⃣ Smoke test: 2 epochs of B0 🚀
Makes sure training, checkpointing and logging run before the long jobs. Delete its results afterwards so they are not mistaken for the real run.

In [ ]:
!python -m src.train --config configs/efficientnet.yaml --epochs 2
shutil.rmtree("results/efficientnet", ignore_errors=True)
Path("saved_models/efficientnet_best.weights.h5").unlink(missing_ok=True)
print("🧹 smoke-test outputs removed")

## 9️⃣ Train the real models 🏋️
Each run: up to 30 epochs, AdamW + cosine LR, weighted BCE (rare diseases count more), early stopping on **validation macro AUC**, best weights saved. After each run, results go to Drive 💾.

📖 **How to read the log:** `auc` / `val_auc` should rise; `loss` / `val_loss` should fall. If train AUC keeps rising while val AUC drops → **overfitting** (early stopping will catch it).

In [ ]:
def backup(run):
    for src_dir, dst in [(f"results/{run}", f"{DRIVE_DIR}/results/{run}"), ("saved_models", f"{DRIVE_DIR}/saved_models")]:
        if Path(src_dir).exists():
            shutil.copytree(src_dir, dst, dirs_exist_ok=True)
    print(f"💾 {run} backed up to Drive")

def restore(run):
    # After a Colab disconnect: bring a finished run back instead of retraining it.
    if Path(f"{DRIVE_DIR}/results/{run}/training_summary.json").exists() and not Path(f"results/{run}").exists():
        shutil.copytree(f"{DRIVE_DIR}/results/{run}", f"results/{run}")
        Path("saved_models").mkdir(exist_ok=True)
        shutil.copy(f"{DRIVE_DIR}/saved_models/{run}_best.weights.h5", "saved_models/")
        print(f"♻️ {run} restored from Drive")
    return Path(f"results/{run}/training_summary.json").exists()

In [ ]:
# 🟩 EfficientNet-B0 (official comparison entry)
if not restore("efficientnet"):
    !python -m src.train --config configs/efficientnet.yaml
backup("efficientnet")

In [ ]:
# 🟩 EfficientNet-B3 at the shared 224 px
if not restore("efficientnet_b3"):
    !python -m src.train --config configs/efficientnet_b3.yaml
backup("efficientnet_b3")

### 🧪 Optional ablation: B3 at its native 300 px
Skip this if time is short. It builds a separate 300 px cache (first epoch is slow). Compare it **only** with B3 @ 224, never with the other models.

In [ ]:
RUN_ABLATION = False   # 👈 set True to run it
if RUN_ABLATION:
    if not restore("efficientnet_b3_300"):
        !python -m src.train --config configs/efficientnet_b3_300.yaml
    backup("efficientnet_b3_300")

## 🔟 Learning curves 📈

In [ ]:
from IPython.display import Image as IPImage, display, Markdown
RUNS = [r for r in ["efficientnet", "efficientnet_b3", "efficientnet_b3_300"] if Path(f"results/{r}/history.csv").exists()]
for r in RUNS:
    s = json.load(open(f"results/{r}/training_summary.json"))
    print(f"{r:20s} best val AUC {s['best_val_auc']:.4f} at epoch {s['best_epoch']}/{s['epochs_run']} | "
          f"{s['mean_epoch_time_s']:.0f} s/epoch | {s['total_params']/1e6:.1f} M params")
    display(IPImage(f"results/{r}/training_curves.png", width=800))

## 1️⃣1️⃣ Validation predictions + threshold tuning 🎚️
The default cut-off is 0.5. For a rare disease like Pneumonia the model may *rank* sick patients well (good AUC) but rarely output > 0.5, so recall is ~0. We pick **one threshold per disease on the validation set** (never on test!).

⚠️ Validation has only ~10 Pneumonia cases, so its tuned threshold is noisy. The **0.5 results stay the official comparison numbers**; tuned results are an extra in the report.

In [ ]:
CFG = {"efficientnet": "configs/efficientnet.yaml", "efficientnet_b3": "configs/efficientnet_b3.yaml",
       "efficientnet_b3_300": "configs/efficientnet_b3_300.yaml"}
for r in RUNS:
    !python -m src.evaluate --config {CFG[r]} --split val

## 1️⃣2️⃣ The test set — ONCE 🧾
🔒 Run this only when training is final. If you look at test results and then change the model, the test set is no longer an honest estimate (it has become a second validation set).

In [ ]:
for r in RUNS:
    !python -m src.evaluate --config {CFG[r]} --split test
    !python -m src.tune_thresholds --config {CFG[r]}
    backup(r)

In [ ]:
for r in RUNS:
    t, tt = json.load(open(f"results/{r}/test_metrics.json")), json.load(open(f"results/{r}/test_metrics_tuned.json"))
    display(Markdown(f"### {r}\nmacro AUC **{t['macro_roc_auc']:.3f}** · macro F1 {t['macro_f1']:.3f} @0.5 → "
                     f"**{tt['macro_f1']:.3f}** tuned · {t['inference_ms_per_image']:.1f} ms/image"))
    display(pd.read_csv(f"results/{r}/test_per_class_metrics.csv").round(3))
    display(IPImage(f"results/{r}/test_roc_curves.png", width=550))

📖 **Reading the numbers**
- **ROC-AUC** (0.5 = coin flip, 1.0 = perfect): how well the model *ranks* sick above healthy, independent of the threshold. The fairest single number for imbalanced medical data.
- **Recall** = of the truly sick, how many we caught (missing a disease is costly in medicine).
- **Precision** = of those we flagged, how many were really sick.
- **Accuracy is misleading here:** predicting "no Pneumonia" for everyone is ~98 % accurate and completely useless.

## 1️⃣3️⃣ Grad-CAM + B0 vs B3 report figures 🔥🏁
Grad-CAM colours the regions that pushed the prediction for the **true** disease. Good sign: heat inside the lungs/heart. Warning sign: heat on text markers, tubes or image borders (a shortcut, not medicine).

In [ ]:
!python -m src.explain --config configs/efficientnet.yaml --num-images 8
!python -m src.efficientnet_report --runs {" ".join(RUNS)} --num-images 7
for f in ["scoreboard.png", "b0_vs_b3_curves.png", "per_class_auc.png", "gradcam_b0_vs_b3.png"]:
    p = Path("results/efficientnet_report") / f
    if p.exists():
        display(Markdown(f"**{f}**")); display(IPImage(str(p), width=850))

## 1️⃣4️⃣ Save everything 💾

In [ ]:
shutil.copytree("results", f"{DRIVE_DIR}/results", dirs_exist_ok=True)
shutil.make_archive("/content/efficientnet_results", "zip", REPO_DIR, "results")
from google.colab import files
files.download("/content/efficientnet_results.zip")
print("✅ Done – commit results/efficientnet*/ and results/efficientnet_report/ (NOT saved_models/) to your branch")

## 🎉 Your EfficientNet part is complete (A → Z)

| ✅ You now have | 📍 Where |
|---|---|
| Trained B0 and B3 (+ optional B3 @ 300) | `saved_models/` (Drive) |
| Curves, test metrics, ROC, confusion matrices | `results/efficientnet*/` |
| Validation-tuned thresholds + tuned test metrics | `thresholds.csv`, `test_metrics_tuned.json` |
| Scoreboard, B0 vs B3 curves, per-disease AUC, Grad-CAM | `results/efficientnet_report/` |

### ✍️ What to write in the report
1. **Compound scaling:** did B3 (2.7× params) beat B0? By how much AUC per extra million parameters and ms?
2. **Resolution ablation (if run):** did 300 px help small findings like *Nodule* and *Mass* more than *Cardiomegaly*?
3. **Fine-tuning choice:** BatchNorm frozen, all conv layers trained (why: small batches would corrupt ImageNet BN statistics).
4. **Limitations:** few Pneumonia/Cardiomegaly test cases → wide uncertainty; NLP-mined labels are noisy.